In [1]:
!git clone https://github.com/anilegin/nlp-sentenceSplitter.git

fatal: destination path 'nlp-sentenceSplitter' already exists and is not an empty directory.


In [3]:
!cd nlp-sentenceSplitter/

In [5]:
from pathlib import Path
from collections import defaultdict

import pandas as pd
import spacy
import nltk

from sklearn.metrics import precision_recall_fscore_support

In [6]:
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [7]:
!python -m spacy download en_core_web_sm
!python -m spacy download it_core_news_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
  Using cached https://github.com/explosion/spacy-models/releases/download/it_core_news_sm-3.8.0/it_core_news_sm-3.8.0-py3-none-any.whl (13.0 MB)
✔ Download and installation successful
You can now load the package via spacy.load('it_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [19]:
nlp_en = spacy.load("en_core_web_sm")
nlp_it = spacy.load("it_core_news_sm")

nlp_en.max_length = 20000000
nlp_it.max_length = 20000000

tokenizer_en = nltk.data.load("tokenizers/punkt/english.pickle")
tokenizer_it = nltk.data.load("tokenizers/punkt/italian.pickle")

In [20]:
#Helper functions

RAW_ROOT = Path("/content/nlp-sentenceSplitter/data/raw")
raw_files = sorted(RAW_ROOT.glob("*/*.sent_split"))

print(f"total files: {len(raw_files)}")
for p in raw_files:
    print(p)

EOS_TOKEN = "<EOS>"


def raw_to_clean_and_gold_boundaries(raw_text, eos_token=EOS_TOKEN):
    clean_chars = []
    gold_boundaries = []

    i = 0
    while i < len(raw_text):
        if raw_text.startswith(eos_token, i):
            gold_boundaries.append(len(clean_chars))
            i += len(eos_token)
        else:
            clean_chars.append(raw_text[i])
            i += 1

    clean_text = "".join(clean_chars)
    return clean_text, gold_boundaries

def predict_boundaries_spacy(text, nlp):
    doc = nlp(text)
    return [sent.end_char for sent in doc.sents]


def predict_boundaries_nltk(text, tokenizer):
    spans = list(tokenizer.span_tokenize(text))
    return [end for _, end in spans]

def evaluate_boundary_sets(gold_boundaries, pred_boundaries):
    gold_set = set(gold_boundaries)
    pred_set = set(pred_boundaries)

    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

def get_language_from_path(path: Path):
    name = path.parent.name.lower()
    if "english" in name:
        return "english"
    if "italian" in name:
        return "italian"
    return "unknown"

def run_sentence_split_baseline(raw_path, language):
    raw_text = raw_path.read_text(encoding="utf-8")
    clean_text, gold_boundaries = raw_to_clean_and_gold_boundaries(raw_text)

    if language == "english":
        pred_spacy = predict_boundaries_spacy(clean_text, nlp_en)
        pred_nltk = predict_boundaries_nltk(clean_text, tokenizer_en)
    elif language == "italian":
        pred_spacy = predict_boundaries_spacy(clean_text, nlp_it)
        pred_nltk = predict_boundaries_nltk(clean_text, tokenizer_it)
    else:
        raise ValueError(f"unknown language for {raw_path}")

    metrics_spacy = evaluate_boundary_sets(gold_boundaries, pred_spacy)
    metrics_nltk = evaluate_boundary_sets(gold_boundaries, pred_nltk)

    return {
        "file": raw_path.name,
        "dataset": raw_path.parent.name,
        "language": language,
        "n_gold": len(gold_boundaries),
        "spacy": metrics_spacy,
        "nltk": metrics_nltk,
        "clean_text_len": len(clean_text),
    }

total files: 22
/content/nlp-sentenceSplitter/data/raw/UD_English-EWT/en_ewt-ud-dev.sent_split
/content/nlp-sentenceSplitter/data/raw/UD_English-EWT/en_ewt-ud-test.sent_split
/content/nlp-sentenceSplitter/data/raw/UD_English-EWT/en_ewt-ud-train.sent_split
/content/nlp-sentenceSplitter/data/raw/UD_English-GUM/en_gum-ud-dev.sent_split
/content/nlp-sentenceSplitter/data/raw/UD_English-GUM/en_gum-ud-test.sent_split
/content/nlp-sentenceSplitter/data/raw/UD_English-GUM/en_gum-ud-train.sent_split
/content/nlp-sentenceSplitter/data/raw/UD_English-PUD/en_pud-ud-test.sent_split
/content/nlp-sentenceSplitter/data/raw/UD_English-ParTUT/en_partut-ud-dev.sent_split
/content/nlp-sentenceSplitter/data/raw/UD_English-ParTUT/en_partut-ud-test.sent_split
/content/nlp-sentenceSplitter/data/raw/UD_English-ParTUT/en_partut-ud-train.sent_split
/content/nlp-sentenceSplitter/data/raw/UD_Italian-ISDT/it_isdt-ud-dev.sent_split
/content/nlp-sentenceSplitter/data/raw/UD_Italian-ISDT/it_isdt-ud-test.sent_split
/co

In [21]:
all_results = []

for raw_path in raw_files:
    language = get_language_from_path(raw_path)
    if language == "unknown":
        continue

    result = run_sentence_split_baseline(raw_path, language)
    all_results.append(result)

    print(
        f"{result['dataset']} | {result['file']} | "
        f"spaCy F1={result['spacy']['f1']:.4f} | "
        f"NLTK F1={result['nltk']['f1']:.4f}"
    )

UD_English-EWT | en_ewt-ud-dev.sent_split | spaCy F1=0.5529 | NLTK F1=0.8323
UD_English-EWT | en_ewt-ud-test.sent_split | spaCy F1=0.5258 | NLTK F1=0.8028
UD_English-EWT | en_ewt-ud-train.sent_split | spaCy F1=0.7104 | NLTK F1=0.8746
UD_English-GUM | en_gum-ud-dev.sent_split | spaCy F1=0.5788 | NLTK F1=0.9291
UD_English-GUM | en_gum-ud-test.sent_split | spaCy F1=0.5725 | NLTK F1=0.9186
UD_English-GUM | en_gum-ud-train.sent_split | spaCy F1=0.5533 | NLTK F1=0.9213
UD_English-PUD | en_pud-ud-test.sent_split | spaCy F1=0.5681 | NLTK F1=0.9910
UD_English-ParTUT | en_partut-ud-dev.sent_split | spaCy F1=0.9342 | NLTK F1=0.9608
UD_English-ParTUT | en_partut-ud-test.sent_split | spaCy F1=0.9216 | NLTK F1=0.9934
UD_English-ParTUT | en_partut-ud-train.sent_split | spaCy F1=0.8935 | NLTK F1=0.9606
UD_Italian-ISDT | it_isdt-ud-dev.sent_split | spaCy F1=0.9267 | NLTK F1=0.9316
UD_Italian-ISDT | it_isdt-ud-test.sent_split | spaCy F1=0.9245 | NLTK F1=0.9539
UD_Italian-ISDT | it_isdt-ud-train.sent_spl

In [22]:
rows = []

for r in all_results:
    rows.append({
        "dataset": r["dataset"],
        "file": r["file"],
        "language": r["language"],
        "n_gold": r["n_gold"],

        "spacy_precision": r["spacy"]["precision"],
        "spacy_recall": r["spacy"]["recall"],
        "spacy_f1": r["spacy"]["f1"],

        "nltk_precision": r["nltk"]["precision"],
        "nltk_recall": r["nltk"]["recall"],
        "nltk_f1": r["nltk"]["f1"],
    })

df_file_results = pd.DataFrame(rows).sort_values(["language", "dataset", "file"]).reset_index(drop=True)
df_file_results

,dataset,file,language,n_gold,spacy_precision,spacy_recall,spacy_f1,nltk_precision,nltk_recall,nltk_f1
0,UD_English-EWT,en_ewt-ud-dev.sent_split,english,2001,0.620261,0.498751,0.552909,0.971333,0.728136,0.832334
1,UD_English-EWT,en_ewt-ud-test.sent_split,english,2077,0.611359,0.461242,0.525796,0.983252,0.678382,0.802849
2,UD_English-EWT,en_ewt-ud-train.sent_split,english,12544,0.762314,0.665019,0.710350,0.971654,0.795201,0.874616
3,UD_English-GUM,en_gum-ud-dev.sent_split,english,1575,0.592420,0.565714,0.578759,0.970934,0.890794,0.929139
4,UD_English-GUM,en_gum-ud-test.sent_split,english,1464,0.593979,0.552596,0.572541,0.971081,0.871585,0.918647
5,UD_English-GUM,en_gum-ud-train.sent_split,english,10224,0.569138,0.538243,0.553260,0.970451,0.876956,0.921338
6,UD_English-PUD,en_pud-ud-test.sent_split,english,1000,0.567298,0.569000,0.568148,0.990020,0.992000,0.991009
7,UD_English-ParTUT,en_partut-ud-dev.sent_split,english,156,0.959459,0.910256,0.934211,0.980000,0.942308,0.960784
8,UD_English-ParTUT,en_partut-ud-test.sent_split,english,153,0.921569,0.921569,0.921569,1.000000,0.986928,0.993421
9,UD_English-ParTUT,en_partut-ud-train.sent_split,english,1781,0.921791,0.866929,0.893519,0.992810,0.930376,0.960580


In [24]:
summary_path = "nltk_spacy_results.csv"
df_file_results.to_csv(summary_path, index=False, encoding="utf-8")

print("saved summary:", summary_path)

saved summary: nltk_spacy_results.csv
